# 🤚 Hand Tracking Desktop Controller
**Auteur : Jean-Marie DJOGBEHOUE**  
**OS : Windows | Python 3.12 ✅ | MediaPipe 0.10.14 · OpenCV · PyAutoGUI**

---
## 🎯 Gestes reconnus

| Geste | Doigts levés | Action |
|---|---|---|
| ☝️ **Index pointé** | index seul | Déplace la souris |
| 🖐️ **Main ouverte** | 4 ou 5 doigts | Clic simple |
| ✊ **Poing fermé** | 0 doigt | Double-clic |

> **🛑 Arrêt :** appuyez sur `Q` dans la fenêtre caméra

## 📦 Cellule 1 — Vérification des dépendances

In [1]:
# Les dépendances sont déjà installées via le terminal.
# Cette cellule vérifie simplement que tout est accessible.

import cv2
import mediapipe as mp
import pyautogui
import numpy as np
import time

# Vérification de l'API mp.solutions
assert hasattr(mp, 'solutions'), "❌ mp.solutions absent — mauvaise version de mediapipe"
assert hasattr(mp.solutions, 'hands'), "❌ mp.solutions.hands absent"

print('✅ Toutes les dépendances sont prêtes !')
print(f'   opencv    : {cv2.__version__}')
print(f'   mediapipe : {mp.__version__}')
print(f'   numpy     : {np.__version__}')

✅ Toutes les dépendances sont prêtes !
   opencv    : 4.13.0
   mediapipe : 0.10.14
   numpy     : 2.4.3


## 🔧 Cellule 2 — Configuration globale

In [2]:
# ── Sécurité PyAutoGUI ──────────────────────────────────────────
pyautogui.FAILSAFE = True   # souris coin haut-gauche = arrêt d'urgence
pyautogui.PAUSE    = 0.0

# ── Résolution écran ────────────────────────────────────────────
SCREEN_W, SCREEN_H = pyautogui.size()

# ── Paramètres (modifiables dans la Cellule 8) ──────────────────
CAMERA_INDEX   = None
CAM_W          = 1280
CAM_H          = 720
SMOOTHING      = 0.18
CLICK_COOLDOWN = 0.75
DETECTION_CONF = 0.75
MARGE          = 80

# Variables d'état
prev_x, prev_y = 0, 0
dernier_clic   = 0
dernier_geste  = None

print(f'✅ Configuration chargée')
print(f'🖥️  Écran : {SCREEN_W} x {SCREEN_H} px')
print(f'🛑 Failsafe ON — souris coin haut-gauche pour stopper')

✅ Configuration chargée
🖥️  Écran : 1600 x 900 px
🛑 Failsafe ON — souris coin haut-gauche pour stopper


## 📷 Cellule 3 — Détection automatique de la webcam

In [3]:
def detecter_webcam():
    print('🔍 Recherche de webcam...')
    for i in range(5):
        cap = cv2.VideoCapture(i, cv2.CAP_DSHOW)
        if cap.isOpened():
            ret, frame = cap.read()
            cap.release()
            if ret and frame is not None:
                print(f'✅ Webcam trouvée — index {i}')
                return i
        cap.release()
    print('❌ Aucune webcam détectée.')
    return None


CAMERA_INDEX = detecter_webcam()

if CAMERA_INDEX is not None:
    cap_test = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
    ret, frame = cap_test.read()
    cap_test.release()
    if ret:
        h, w = frame.shape[:2]
        print(f'📐 Résolution native : {w}x{h} px')
        print(f'🎯 Prêt — index = {CAMERA_INDEX}')

🔍 Recherche de webcam...
✅ Webcam trouvée — index 0
📐 Résolution native : 640x480 px
🎯 Prêt — index = 0


## 🤚 Cellule 4 — Initialisation MediaPipe & reconnaissance des gestes

In [4]:
# ── Initialisation MediaPipe Hands ──────────────────────────────
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

try:
    mp_styles  = mp.solutions.drawing_styles
    HAS_STYLES = True
except AttributeError:
    mp_styles  = None
    HAS_STYLES = False
    print('⚠️  drawing_styles non disponible, squelette simplifié.')

# Indices des landmarks (21 points par main)
BOUT_POUCE       = 4
BOUT_INDEX       = 8
BOUT_MAJEUR      = 12
BOUT_ANNULAIRE   = 16
BOUT_AURICULAIRE = 20
JOIN_INDEX       = 6
JOIN_MAJEUR      = 10
JOIN_ANNULAIRE   = 14
JOIN_AURICULAIRE = 18


def compter_doigts(lm):
    """
    Compte les doigts levés à partir des landmarks MediaPipe.
    Retourne un entier entre 0 (poing) et 5 (main ouverte).
    """
    doigts = 0
    # Pouce (axe X)
    if lm[BOUT_POUCE].x < lm[3].x:
        doigts += 1
    # 4 autres doigts (axe Y)
    for bout, join in [
        (BOUT_INDEX,       JOIN_INDEX),
        (BOUT_MAJEUR,      JOIN_MAJEUR),
        (BOUT_ANNULAIRE,   JOIN_ANNULAIRE),
        (BOUT_AURICULAIRE, JOIN_AURICULAIRE),
    ]:
        if lm[bout].y < lm[join].y:
            doigts += 1
    return doigts


def detecter_geste(lm):
    """
    Identifie le geste à partir des landmarks.
    Retourne : ('pointer' | 'main_ouverte' | 'poing' | 'neutre', nb_doigts)
    """
    nb = compter_doigts(lm)

    index_leve      = lm[BOUT_INDEX].y       < lm[JOIN_INDEX].y
    majeur_leve     = lm[BOUT_MAJEUR].y      < lm[JOIN_MAJEUR].y
    annulaire_leve  = lm[BOUT_ANNULAIRE].y   < lm[JOIN_ANNULAIRE].y
    auriculaire_leve= lm[BOUT_AURICULAIRE].y < lm[JOIN_AURICULAIRE].y

    # ☝️ Index seul levé
    if index_leve and not majeur_leve and not annulaire_leve and not auriculaire_leve:
        return 'pointer', nb

    # 🖐️ Main ouverte
    elif nb >= 4:
        return 'main_ouverte', nb

    # ✊ Poing
    elif nb <= 1:
        return 'poing', nb

    else:
        return 'neutre', nb


print('✅ MediaPipe initialisé — mp.solutions.hands OK')
print('✅ Fonctions detecter_geste() et compter_doigts() chargées.')

✅ MediaPipe initialisé — mp.solutions.hands OK
✅ Fonctions detecter_geste() et compter_doigts() chargées.


## 🖱️ Cellule 5 — Contrôle de la souris

In [5]:
def index_vers_ecran(lm, cam_w, cam_h):
    """Convertit la position normalisée de l'index en coordonnées écran."""
    x_norm = lm[BOUT_INDEX].x
    y_norm = lm[BOUT_INDEX].y

    x_adj = (x_norm - MARGE / cam_w) / (1 - 2 * MARGE / cam_w)
    y_adj = (y_norm - MARGE / cam_h) / (1 - 2 * MARGE / cam_h)
    x_adj = max(0.0, min(1.0, x_adj))
    y_adj = max(0.0, min(1.0, y_adj))

    # Inversion miroir
    x_ecran = int((1 - x_adj) * SCREEN_W)
    y_ecran = int(y_adj * SCREEN_H)
    return x_ecran, y_ecran


def deplacer_souris(x_cible, y_cible):
    global prev_x, prev_y
    x = int(prev_x + (x_cible - prev_x) * (1 - SMOOTHING))
    y = int(prev_y + (y_cible - prev_y) * (1 - SMOOTHING))
    pyautogui.moveTo(x, y)
    prev_x, prev_y = x, y


def executer_action(geste, x_ecran, y_ecran):
    global dernier_clic, dernier_geste
    now = time.time()

    if geste == 'pointer':
        deplacer_souris(x_ecran, y_ecran)

    elif geste == 'main_ouverte':
        if now - dernier_clic > CLICK_COOLDOWN and dernier_geste != 'main_ouverte':
            pyautogui.click()
            dernier_clic = now
            print(f'🖱️  CLIC          ({x_ecran}, {y_ecran})')

    elif geste == 'poing':
        if now - dernier_clic > CLICK_COOLDOWN and dernier_geste != 'poing':
            pyautogui.doubleClick()
            dernier_clic = now
            print(f'🖱️🖱️ DOUBLE-CLIC  ({x_ecran}, {y_ecran})')

    dernier_geste = geste


print('✅ Fonctions souris chargées.')

✅ Fonctions souris chargées.


## 🎨 Cellule 6 — Overlay visuel

In [6]:
C_VERT   = (50,  220,  80)
C_ORANGE = (30,  160, 255)
C_ROUGE  = (60,   60, 230)
C_GRIS   = (160, 160, 160)
C_BLANC  = (255, 255, 255)
C_FOND   = (18,   22,  32)

GESTE_INFO = {
    'pointer':      ('☝  INDEX  — Deplacement souris', C_VERT),
    'main_ouverte': ('MAIN OUVERTE  — Clic simple',    C_ORANGE),
    'poing':        ('POING FERME  — Double-clic',     C_ROUGE),
    'neutre':       ('Geste neutre...',                C_GRIS),
    None:           ('En attente de main...',          C_GRIS),
}


def dessiner_overlay(frame, geste, nb_doigts, lm_pixel=None):
    h, w = frame.shape[:2]
    texte, couleur = GESTE_INFO.get(geste, GESTE_INFO[None])

    # Bandeau haut
    cv2.rectangle(frame, (0, 0), (w, 58), C_FOND, -1)
    cv2.putText(frame, texte, (14, 38),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, couleur, 2, cv2.LINE_AA)
    cv2.putText(frame, f'{nb_doigts} doigt(s)', (w - 165, 38),
                cv2.FONT_HERSHEY_SIMPLEX, 0.78, C_BLANC, 2, cv2.LINE_AA)

    # Viseur sur l'index
    if lm_pixel and geste == 'pointer':
        ix, iy = lm_pixel
        cv2.circle(frame, (ix, iy), 16, couleur, -1)
        cv2.circle(frame, (ix, iy), 16, C_BLANC,  2)
        cv2.line(frame, (ix, iy-22), (ix, iy-44), couleur, 2)
        cv2.line(frame, (ix-22, iy), (ix-44, iy), couleur, 2)
        cv2.line(frame, (ix+22, iy), (ix+44, iy), couleur, 2)

    # Bandeau bas
    cv2.rectangle(frame, (0, h-36), (w, h), C_FOND, -1)
    cv2.putText(frame, 'Q = Quitter   |   Auteur : Jean-Marie DJOGBEHOUE',
                (10, h-10), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (100,110,130), 1, cv2.LINE_AA)

    # Bordure colorée
    cv2.rectangle(frame, (0, 0), (w-1, h-1), couleur, 3)

    return frame


print('✅ Overlay chargé.')

✅ Overlay chargé.


---
## 🚀 Cellule 7 — LANCER LE CONTRÔLEUR

> Une fenêtre s'ouvre avec le flux caméra en direct.  
> **Appuyez sur `Q`** dans la fenêtre caméra pour arrêter.

In [ ]:
if CAMERA_INDEX is None:
    print('❌ Webcam non détectée — relancez la Cellule 3.')
else:
    print('🚀 Démarrage du Hand Tracking Controller...')
    print('👋 Montrez votre MAIN DROITE devant la caméra !')
    print('🛑 Appuyez sur Q dans la fenêtre pour arrêter.\n')

    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  CAM_W)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAM_H)
    cap.set(cv2.CAP_PROP_FPS, 30)

    prev_x, prev_y = SCREEN_W // 2, SCREEN_H // 2
    dernier_clic   = 0
    dernier_geste  = None

    with mp_hands.Hands(
        max_num_hands            = 1,
        min_detection_confidence = DETECTION_CONF,
        min_tracking_confidence  = DETECTION_CONF,
    ) as hands:

        print('📷 Caméra ouverte — fenêtre en cours d\'affichage...')

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            cam_h, cam_w = frame.shape[:2]

            # ── Traitement MediaPipe ──────────────────────────────
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_rgb.flags.writeable = False
            resultats = hands.process(frame_rgb)
            frame_rgb.flags.writeable = True
            frame = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

            geste     = None
            nb_doigts = 0
            lm_pixel  = None

            if resultats.multi_hand_landmarks:
                for hand_lm in resultats.multi_hand_landmarks:

                    # Dessiner le squelette de la main
                    if HAS_STYLES:
                        mp_drawing.draw_landmarks(
                            frame, hand_lm, mp_hands.HAND_CONNECTIONS,
                            mp_styles.get_default_hand_landmarks_style(),
                            mp_styles.get_default_hand_connections_style()
                        )
                    else:
                        mp_drawing.draw_landmarks(
                            frame, hand_lm, mp_hands.HAND_CONNECTIONS,
                            mp_drawing.DrawingSpec(color=(0,220,80),  thickness=2, circle_radius=4),
                            mp_drawing.DrawingSpec(color=(255,255,255), thickness=2)
                        )

                    lm = hand_lm.landmark

                    # Position pixel du bout de l'index
                    lm_pixel = (
                        int(lm[BOUT_INDEX].x * cam_w),
                        int(lm[BOUT_INDEX].y * cam_h)
                    )

                    # Geste + coordonnées écran
                    geste, nb_doigts = detecter_geste(lm)
                    x_ecran, y_ecran = index_vers_ecran(lm, cam_w, cam_h)
                    executer_action(geste, x_ecran, y_ecran)

            frame = dessiner_overlay(frame, geste, nb_doigts, lm_pixel)
            cv2.imshow('Hand Tracking Controller — Jean-Marie DJOGBEHOUE', frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                print('\n🛑 Arrêt demandé.')
                break

    cap.release()
    cv2.destroyAllWindows()
    print('✅ Contrôleur arrêté proprement.')

---
## ⚙️ Cellule 8 — Réglages fins (relancer Cellule 7 après modification)

In [ ]:
# Tremblements ?          → augmenter SMOOTHING      (ex: 0.30)
# Trop lent ?             → diminuer  SMOOTHING      (ex: 0.08)
# Clics involontaires ?   → augmenter CLICK_COOLDOWN (ex: 1.2)
# Mauvaise détection ?    → diminuer  DETECTION_CONF (ex: 0.60)
# Souris hors écran ?     → augmenter MARGE          (ex: 120)
# Lag / lenteur ?         → réduire   CAM_W=640, CAM_H=480

SMOOTHING      = 0.18
CLICK_COOLDOWN = 0.75
DETECTION_CONF = 0.70
MARGE          = 80
CAM_W, CAM_H   = 1280, 720

print('✅ Réglages mis à jour — relancez la Cellule 7.')

---
## 📋 Référence rapide

| Problème | Solution |
|---|---|
| Souris tremble | `SMOOTHING = 0.30` |
| Main mal détectée | `DETECTION_CONF = 0.60` |
| Clics involontaires | `CLICK_COOLDOWN = 1.2` |
| Souris sort de l'écran | `MARGE = 120` |
| Lag / lenteur | `CAM_W=640, CAM_H=480` |
| Poing mal reconnu | Fermer bien tous les doigts |

---
**Auteur : Jean-Marie DJOGBEHOUE** — Python 3.12 · MediaPipe 0.10.14 · OpenCV · PyAutoGUI